In [1]:
# Image Stitching and Panorama Creation
# Manual pipeline + OpenCV Stitcher

import cv2
import numpy as np
import matplotlib.pyplot as plt
from typing import List
import os
from pathlib import Path

# ------------------------------------------------
# 0. Configuration
# ------------------------------------------------

IMAGE_FOLDER = "images/panorama_set"
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png"]


# ------------------------------------------------
# Helper Functions
# ------------------------------------------------

def load_images(folder: str) -> List[np.ndarray]:
    """Load all images from folder"""
    
    images = []

    for file in sorted(os.listdir(folder)):
        if Path(file).suffix.lower() in IMAGE_EXTENSIONS:
            img_path = os.path.join(folder, file)

            img = cv2.imread(img_path)

            if img is not None:
                images.append(img)
                print(f"Loaded: {file}  shape: {img.shape}")

    return images


def show_images(images: List[np.ndarray], titles=None, figsize=(15,6)):
    """Display images side by side"""

    n = len(images)

    plt.figure(figsize=figsize)

    for i, img in enumerate(images):

        plt.subplot(1, n, i+1)
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

        if titles:
            plt.title(titles[i])
        else:
            plt.title(f"Image {i+1}")

        plt.axis("off")

    plt.tight_layout()
    plt.show()

# ------------------------------------------------
# 🔥 NEW PART: SHOW FEATURE MATCHES (VISUALIZATION)
# ------------------------------------------------

def show_feature_matches(img1, img2, kp1, kp2, matches, title="Feature Matches"):

    # Sort best matches
    matches = sorted(matches, key=lambda x: x.distance)

    # Create blank canvas (side by side)
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]

    canvas = np.zeros((max(h1, h2), w1 + w2, 3), dtype=np.uint8)
    canvas[:h1, :w1] = img1
    canvas[:h2, w1:w1+w2] = img2

    # Bright neon colors
    colors = [
        (255, 0, 0),     # Blue
        (0, 255, 0),     # Green
        (0, 0, 255),     # Red
        (255, 255, 0),   # Cyan
        (255, 0, 255),   # Magenta
        (0, 255, 255),   # Yellow
    ]

    # Draw matches
    for i, m in enumerate(matches[:80]):  # top 80 matches

        color = colors[i % len(colors)]

        pt1 = tuple(map(int, kp1[m.queryIdx].pt))
        pt2 = tuple(map(int, kp2[m.trainIdx].pt))

        pt2_shifted = (int(pt2[0] + w1), int(pt2[1]))

        # 🔥 Thick bright line
        cv2.line(canvas, pt1, pt2_shifted, color, 2)

        # 🔥 Big bright circles
        cv2.circle(canvas, pt1, 5, color, -1)
        cv2.circle(canvas, pt2_shifted, 5, color, -1)

    # Show result
    plt.figure(figsize=(18,8))
    plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    plt.title(f"{title} ({len(matches)} matches)", fontsize=14)
    plt.axis("off")
    plt.show()
# ------------------------------------------------
# 1. Feature Detection + Matching
# ------------------------------------------------

def detect_and_match_features(img1, img2):

    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

    orb = cv2.ORB_create(nfeatures=2000)

    kp1, des1 = orb.detectAndCompute(gray1, None)
    kp2, des2 = orb.detectAndCompute(gray2, None)

    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)

    matches = bf.knnMatch(des1, des2, k=2)

    good_matches = []

    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            good_matches.append(m)

    return kp1, des1, kp2, des2, good_matches


# ------------------------------------------------
# 2. Homography Estimation
# ------------------------------------------------

def compute_homography(kp1, kp2, good_matches, min_matches=10):

    if len(good_matches) < min_matches:
        return None, None

    src_pts = np.float32(
        [kp1[m.queryIdx].pt for m in good_matches]
    ).reshape(-1,1,2)

    dst_pts = np.float32(
        [kp2[m.trainIdx].pt for m in good_matches]
    ).reshape(-1,1,2)

    H, mask = cv2.findHomography(
        src_pts,
        dst_pts,
        cv2.RANSAC,
        5.0
    )

    return H, mask


# ------------------------------------------------
# 3. Warp + Blend
# ------------------------------------------------

def stitch_two_images(img1, img2, H):

    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]

    corners = np.float32([
        [0,0],
        [0,h2],
        [w2,h2],
        [w2,0]
    ]).reshape(-1,1,2)

    warped_corners = cv2.perspectiveTransform(corners, H)

    all_corners = np.concatenate((
        np.float32([
            [0,0],
            [0,h1],
            [w1,h1],
            [w1,0]
        ]).reshape(-1,1,2),
        warped_corners
    ), axis=0)

    xmin, ymin = np.int32(all_corners.min(axis=0).ravel() - 0.5)
    xmax, ymax = np.int32(all_corners.max(axis=0).ravel() + 0.5)

    translation = np.array([
        [1,0,-xmin],
        [0,1,-ymin],
        [0,0,1]
    ])

    result = cv2.warpPerspective(
        img2,
        translation.dot(H),
        (xmax-xmin, ymax-ymin)
    )

    result[-ymin:h1-ymin, -xmin:w1-xmin] = img1

    return result


# ------------------------------------------------
# 4. Sequential Stitching
# ------------------------------------------------

def simple_stitch_all(images):

    panorama = images[0].copy()

    for i in range(1, len(images)):

        print(f"Stitching image {i}...")

        kp1, des1, kp2, des2, good = detect_and_match_features(
            panorama,
            images[i]
        )

        if len(good) < 10:
            print("Not enough matches")
            continue

        H, mask = compute_homography(kp1, kp2, good)

        if H is None:
            print("Homography failed")
            continue

        panorama = stitch_two_images(
            panorama,
            images[i],
            H
        )

    return panorama


# ------------------------------------------------
# 5. Load Images
# ------------------------------------------------

images = load_images(IMAGE_FOLDER)

if len(images) < 2:
    raise ValueError("Need at least 2 images")

show_images(
    images[:4],
    titles=[f"Input {i+1}" for i in range(min(4,len(images)))]
)

# ------------------------------------------------
# 🔥 FEATURE MATCHES VISUALIZATION
# ------------------------------------------------

print("\nShowing Feature Matches...")

if len(images) >= 2:

    kp1, des1, kp2, des2, good_matches = detect_and_match_features(
        images[0],
        images[1]
    )

    print(f"Good matches between Image 1 & Image 2: {len(good_matches)}")

    show_feature_matches(
        images[0],
        images[1],
        kp1,
        kp2,
        good_matches,
        title="Matches: Image 1 → Image 2"
    )
    
# ------------------------------------------------
# 7. OpenCV Auto Stitcher
# ------------------------------------------------

print("\nRunning OpenCV Stitcher...")

stitcher = cv2.Stitcher_create(cv2.Stitcher_PANORAMA)

status, pano = stitcher.stitch(images)

if status == cv2.Stitcher_OK:

    print("Stitching successful")

    plt.figure(figsize=(14,8))
    plt.imshow(cv2.cvtColor(pano, cv2.COLOR_BGR2RGB))
    plt.title("OpenCV Panorama")
    plt.axis("off")
    plt.show()

else:

    print("Stitching failed:", status)

# Reload fresh images (VERY IMPORTANT)
images_scans = load_images(IMAGE_FOLDER)

# Resize for stability
resized_scans = []
for img in images_scans:
    h, w = img.shape[:2]
    scale = 0.6
    resized_scans.append(cv2.resize(img, (int(w*scale), int(h*scale))))
# ------------------------------------------------
# 8. SCANS Mode
# ------------------------------------------------

print("\nTrying SCANS mode...")

stitcher_scans = cv2.Stitcher_create(cv2.Stitcher_SCANS)

# Optional tuning
stitcher_scans.setPanoConfidenceThresh(0.5)

status_scans, pano_scans = stitcher_scans.stitch(resized_scans)

print("SCANS Status:", status_scans)

if status_scans == cv2.Stitcher_OK:

    print("✅ SCANS Stitching Successful")

    plt.figure(figsize=(14,8))
    plt.imshow(cv2.cvtColor(pano_scans, cv2.COLOR_BGR2RGB))
    plt.title("SCANS Mode Panorama")
    plt.axis("off")
    plt.show()

else:

    print("❌ SCANS Stitch Failed:", status_scans)

# ------------------------------------------------
# 🔥 FINAL BEST PANORAMA OUTPUT (LIKE YOUR FRIEND)
# ------------------------------------------------

print("\nSelecting Best Final Panorama Output...")

final_pano = None
final_title = ""

# Priority: PANORAMA first
if status == cv2.Stitcher_OK:
    final_pano = pano
    final_title = "OpenCV PANORAMA"

# If panorama failed, use SCANS
elif status_scans == cv2.Stitcher_OK:
    final_pano = pano_scans
    final_title = "OpenCV SCANS"

# If both failed
else:
    print("❌ No successful panorama generated")
    final_pano = None



# Show final output
if final_pano is not None:

    print(f"Best final output selected: {final_title}")

    plt.figure(figsize=(16,8))
    plt.imshow(cv2.cvtColor(final_pano, cv2.COLOR_BGR2RGB))
    plt.title(f"Final Panorama Output ({final_title})", fontsize=14)
    plt.axis("off")
    plt.show()

    # Optional: Save output
    os.makedirs("output_panorama", exist_ok=True)
    save_path = os.path.join("output_panorama", "final_panorama_output.jpg")
    cv2.imwrite(save_path, final_pano)

    print(f"Saved final output at: {save_path}")
    
print("Lab Completed")

FileNotFoundError: [Errno 2] No such file or directory: 'images/panorama_set'